# Climate Trend Analyzer: Exploratory Analysis (Real Public Data)

**Project Author**: Akash Das  
**Dataset**: NOAA Global Historical Climatology Network - Daily (GHCN-Daily)  
**Station**: New York City Central Park (`USW00094728`)  
**Temporal Coverage**: 1869 to Present  
**Reference Climatological Baseline**: 1951–1980 Mean  

This notebook demonstrates empirical climate trend quantification, statistical extreme detection, and out-of-sample temperature forecasting based strictly on genuine, publicly available meteorological observations.

In [ ]:
import sys
from pathlib import Path

# Ensure src/ directory is available on path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from climate_analyzer.ingest import load_climate_dataset
from climate_analyzer.validation import validate_ghcn_dataframe
from climate_analyzer.transform import (
    clean_daily_observations,
    aggregate_to_monthly,
    compute_climatological_anomalies,
    aggregate_to_annual,
)
from climate_analyzer.features import engineer_monthly_features, engineer_annual_features
from climate_analyzer.analysis import calculate_linear_trend, mann_kendall_test, sens_slope
from climate_analyzer.anomaly_detection import detect_anomalies, get_top_extreme_records
from climate_analyzer.modeling import train_and_evaluate_climate_model

## 1. Load Genuine NOAA GHCN-Daily Climate Records
We load genuine observations from the local raw cache.

In [ ]:
raw_dir = project_root / 'data' / 'raw'
daily_df = load_climate_dataset(station_id='USW00094728', raw_dir=raw_dir, allow_download=False)
print(f"Loaded {len(daily_df):,} daily observations spanning {daily_df['date'].min().date()} to {daily_df['date'].max().date()}.")
daily_df[['date', 'tmax', 'tmin', 'tavg', 'prcp']].head()

## 2. Quality Assertion & Schema Validation
Verify column types, temporal ordering, and physical plausibility.

In [ ]:
val_report = validate_ghcn_dataframe(daily_df, raise_on_error=False)
print("Validation Status:", "PASSED" if val_report['is_valid'] else "FAILED")
print("Warnings / Notes:", val_report['warnings'])

## 3. Data Transformation & Climatological Baseline (1951–1980)
Clean isolated observation gaps, aggregate daily series to monthly indicators, and compute monthly anomalies relative to the 1951-1980 baseline.

In [ ]:
clean_daily = clean_daily_observations(daily_df)
monthly_df = aggregate_to_monthly(clean_daily, min_days_per_month=25)
monthly_anom = compute_climatological_anomalies(monthly_df, baseline_start=1951, baseline_end=1980)
annual_df = aggregate_to_annual(monthly_anom, min_months_per_year=11)

annual_feats = engineer_annual_features(annual_df, rolling_windows=(5, 10))
annual_feats.tail()

## 4. Trend Quantification & Hypothesis Testing
Fit Ordinary Least Squares (OLS), calculate Sen's non-parametric slope, and run the Mann-Kendall monotonic trend test.

In [ ]:
trend_ols = calculate_linear_trend(annual_feats['year'], annual_feats['annual_anomaly'])
sens = sens_slope(annual_feats['year'], annual_feats['annual_anomaly'])
mk = mann_kendall_test(annual_feats['annual_anomaly'])

print(f"OLS Linear Warming Rate:  {trend_ols['decadal_rate']:+.3f} °C / decade (p = {trend_ols['p_value']:.2e}, R² = {trend_ols['r_squared']:.3f})")
print(f"Sen's Non-Parametric Slope: {sens['sens_decadal_rate']:+.3f} °C / decade")
print(f"Mann-Kendall Monotonic:     {mk['trend'].upper()} (S = {mk['s']:,}, Z = {mk['z']:.2f}, p = {mk['p_value']:.2e})")

## 5. Statistical Outliers & Meteorological Extremes
Rank the warmest recorded years in the genuine 150+ year station history.

In [ ]:
top_warmest = get_top_extreme_records(annual_feats, value_col='annual_tavg', top_n=10, warmest=True)
top_warmest

## 6. Out-of-Sample Machine Learning Evaluation
Train a Ridge Regressor on historical observations (1870–2000) and evaluate on held-out observations (2001–Present) compared against Climatological and Persistence baselines.

In [ ]:
monthly_feats = engineer_monthly_features(monthly_anom)
eval_res = train_and_evaluate_climate_model(monthly_feats, split_year=2000)

metrics = eval_res['metrics']
print("Evaluation on Held-Out Test Records (2001 - Present):")
print(f"  Ridge ML Model:       RMSE = {metrics['ml_model']['rmse']:.3f} °C, MAE = {metrics['ml_model']['mae']:.3f} °C, R² = {metrics['ml_model']['r2']:.3f}")
print(f"  Climatology Baseline: RMSE = {metrics['climatology_baseline']['rmse']:.3f} °C, MAE = {metrics['climatology_baseline']['mae']:.3f} °C, R² = {metrics['climatology_baseline']['r2']:.3f}")
print(f"  Persistence Baseline: RMSE = {metrics['persistence_baseline']['rmse']:.3f} °C, MAE = {metrics['persistence_baseline']['mae']:.3f} °C, R² = {metrics['persistence_baseline']['r2']:.3f}")